# Qwen3-VL-4B (4-bit) zero-shot on the hardest FinTabNet tables (Colab)

Zero-shot ceiling for the **trainable student** before any fine-tuning. Runs `Qwen/Qwen3-VL-4B-Instruct` in 4-bit through the repo's `TableReconstructor` (transformers backend) over the 50 hardest FinTabNet validation tables (merged cells guaranteed), scored with the same TEDS-Struct + span-recall + bootstrap-CI pipeline as every other candidate.

**Public data only.** FinTabNet is public S&P-500 tables and `MODEL_4B` is `data="both"`, so nothing here touches the confidential-invoice boundary — Colab is fine. The 20 private invoices stay on-prem (Track 2).

**Same 50 tables as `paddle_specialist_colab.ipynb`.** The corpus is built with identical parameters (`n_target=50`, `max_scanned=20000`), so difficulty ranking selects the same tables — this zero-shot student number is directly comparable to the PaddleOCR-VL specialist ceiling.

**Backend = transformers, not unsloth.** A pure zero-shot run needs only the 4-bit transformers path, so this notebook skips `unsloth`/`vllm` and their Colab torch-conflict + runtime-restart dance. One consequence, per CLAUDE.md's load-parity rule: if you later A/B this baseline against an **Unsloth-trained** LoRA, re-run the baseline with `backend="unsloth"` so both arms quantize identically. For a standalone ceiling number, transformers is correct.

Runtime: **GPU** — Runtime ▸ Change runtime type ▸ **T4 GPU** (plenty for a 4B in 4-bit).

## 1. Setup — clone repo + install

`requirements-base.txt` is the CPU-only repo deps (loader, eval). Zero-shot adds only the transformers 4-bit stack; `transformers>=4.57.0` is required for Qwen3-VL (`AutoModelForImageTextToText`).

> If the model class is not found after this cell, Colab had an older `transformers` already imported: **Runtime ▸ Restart**, then re-run from the *Load* cell (§3) — you do **not** need to re-clone or re-install.

In [ ]:
BRANCH = "phase-two-distillation-pipeline"
REPO   = "https://github.com/hidrochin/qwen-vl-table-reconstruction.git"

import os, sys
if not os.path.isdir("qwen-vl-table-reconstruction"):
    !git clone --branch $BRANCH $REPO
%cd qwen-vl-table-reconstruction
!git checkout $BRANCH && git pull --ff-only

# Repo runtime deps (loader pages the datasets-server; eval uses numpy). CPU, fast.
!pip install -q -r requirements-base.txt
# Zero-shot path only: transformers 4-bit. NOT unsloth/vllm (avoids the Colab
# torch-conflict restart). transformers>=4.57 is needed for Qwen3-VL.
!pip install -q -U "transformers>=4.57.0" accelerate bitsandbytes qwen-vl-utils

# Make `import src...` work from the repo root.
sys.path.insert(0, os.getcwd())
print("\ncwd:", os.getcwd())

## 2. Build the 50 hardest-table eval corpus

`build_split` scans the FinTabNet **validation** split, ranks by difficulty, keeps the top 50 tables with at least one span (merged cell — the capability under test), deduplicates by normalized HTML, and downloads only those images (~a few MB). Only HTML is scored during the scan, so paging thousands of rows is cheap.

**Rate limit.** The anonymous `datasets-server` API returns **HTTP 429** under sustained paging, so `max_scanned=5000` scans a portion rather than the whole 20k split (~2k spanning candidates by then — the hardest 50 are well-covered). The loader also paces requests, honors `Retry-After`, and waits out a 429 instead of dying; `HF_TOKEN` lifts the limit further. Keep `max_scanned` identical to `paddle_specialist_colab.ipynb` so both runs score the **same 50 tables**.

In [ ]:
from pathlib import Path
from src.data.loader import build_split

# OPTIONAL but recommended: an HF token lifts the datasets-server rate limit a lot,
# so the scan won't throttle. Get one at https://huggingface.co/settings/tokens
# (read scope is enough) and uncomment:
# import os; os.environ["HF_TOKEN"] = "hf_xxx"

CORPUS = Path("data/corpus")
records = build_split(
    split_name="eval",
    source_split="validation",
    n_target=50,
    out_dir=CORPUS,
    # Scan a portion, not the whole split. The anonymous datasets-server API 429s
    # under sustained paging; 5,000 rows already yields ~2k spanning candidates, so
    # the hardest 50 are well-covered. Must match paddle_specialist_colab.ipynb for
    # the two runs to score the SAME 50 tables. Raise it (or set HF_TOKEN) on a
    # "wanted 50, got N" warning.
    max_scanned=5000,
)
print(f"\nbuilt {len(records)} tables -> {CORPUS/'eval'}")

## 3. Load Qwen3-VL-4B (4-bit, transformers backend)

First construction downloads the ~4B weights and quantizes to 4-bit (nf4). `gpu_report()` prints the card and whether bf16 is available — on a **T4** you'll see `bf16=False`, which is expected; the 4-bit path falls back to fp16 compute automatically.

In [ ]:
from src.model.inference import TableReconstructor, gpu_report, MODEL_4B

print(gpu_report())
model = TableReconstructor(
    model_id=MODEL_4B,
    backend="transformers",   # see title note on load-parity before any A/B
    load_in_4bit=True,
)
print("loaded", MODEL_4B)
print(gpu_report())

## 4. Smoke test — 5 tables, read the raw output

**Do this before the full 50.** The first run of generation is not the moment to discover a bad prompt 20 minutes in. Read the raw output and check three things:

1. It is **HTML** at all (starts heading toward `<table>`).
2. `clean_prediction` found a `<table>` — the cleaned html below is non-empty.
3. It **respected structure mode**: tags and span attributes, **no cell text**. If it emits full numbers/words, the structure prompt is being ignored — stop and investigate before spending time on all 50.

In [ ]:
from src.data.loader import load_manifest

# Re-load so image paths are absolute (build_split returns them relative).
records = load_manifest(CORPUS / "eval")
print(f"{len(records)} eval tables\n")

smoke = model.predict_many([r.image_path for r in records[:5]], mode="structure")
print("=== raw output, table 0 (first 600 chars) ===")
print(smoke[0].raw[:600])
print("\n=== cleaned html, table 0 (first 400 chars) ===")
print(smoke[0].html[:400] or "*** EMPTY — clean_prediction found no <table> ***")

## 5. Predict over all 50 tables

Sequential greedy generation (`do_sample=False`) with progress every 10 tables. On a T4 in 4-bit this is roughly 20–40s per table.

In [ ]:
image_paths = [r.image_path for r in records]
preds = model.predict_many(image_paths, mode="structure")
print(f"\n{len(preds)} predictions; "
      f"{sum(1 for p in preds if not p.html.strip())} empty")
print(gpu_report())

## 6. Score + report

Same pipeline as every candidate: TEDS-Struct (structure-only), position-aware span recall, bootstrap CIs, per-difficulty-bin breakdown, and parse failures.

**Calibration — do not misread the number.** Qwen2.5-VL-**32B** scores 81.7 TEDS zero-shot; a **4B in 4-bit** on deliberately hard tables landing anywhere in **0.55–0.75 TEDS-Struct is a normal result**, not a failure. Published 97% numbers are specialized models trained on the target set. Watch `parse-fails`: a handful is fine, but 20/50 means the prompt or `max_new_tokens` is wrong, not the model.

In [ ]:
from src.eval.runner import evaluate_predictions, save_run
from src.model.inference import predictions_dict

results, summary = evaluate_predictions(
    records, predictions_dict(preds), "qwen3-vl-4b-zeroshot"
)
save_run(results, summary, Path("outputs/runs"))

print(f"=== Qwen3-VL-4B zero-shot over {summary.n} hardest FinTabNet tables ===")
print(f"TEDS-Struct : {summary.mean_teds:.4f}  [{summary.ci_low:.3f}, {summary.ci_high:.3f}]")
print(f"span-recall : {summary.mean_span_recall:.4f}")
print(f"parse-fails : {summary.parse_failures}/{summary.n}")
if summary.by_bin:
    print("\nby difficulty bin:")
    for label, b in summary.by_bin.items():
        print(f"  {label:<7} n={b['n']:<3} TEDS {b['mean']:.4f} [{b['ci'][0]:.3f}, {b['ci'][1]:.3f}]")

## 7. Eyeball the hardest failures

Worst TEDS-Struct cases first. In structure mode the predicted html carries tags only, so compare **shape** (rows, spans, hierarchy) against the ground truth — this is where a real structural miss gets separated from an HTML-dialect penalty.

In [ ]:
from src.eval.runner import to_cases

for c in to_cases(results, limit=3, hardest_first=True):
    print("=" * 70)
    print(f"{c.uid}  TEDS-Struct={c.score:.3f}  difficulty={c.difficulty}  spans={c.n_spanning}")
    print("  image:", c.image_path)
    print("  --- predicted (first 400 chars) ---\n ", c.pred_html[:400])
    print("  --- ground truth (first 400 chars) ---\n ", c.true_html[:400])

## 8. Free VRAM / next steps

`model.close()` releases the weights and empties the CUDA cache — do this before loading any other model in the same session (two on one T4 is an OOM).

Next candidates on the same corpus:

- **8B zero-shot ceiling** — `TableReconstructor(MODEL_8B, load_in_4bit=True)` after `close()`; *4B fine-tuned beating 8B zero-shot* is the compelling outcome.
- **PaddleOCR-VL specialist ceiling** — `paddle_specialist_colab.ipynb`, same 50 tables, then `compare_runs()` for the head-to-head.
- **LoRA fine-tune the 4B** — switch to `backend="unsloth"` here for load-parity, then train (`train.py` / the Unsloth vision notebook).

In [ ]:
model.close()
print(gpu_report())